# QADC: Query-Aware Dynamic Compression for Efficient MLLMs

**Complete experiment notebook** — runs end-to-end on Google Colab (T4 GPU).

### Execution order
| Step | Cell | Time (est.) |
|------|------|------------|
| 0 | Install & mount | 5 min |
| 1 | Download datasets | 15 min |
| 2 | Generate pseudo labels | 60 min (resumable) |
| 3 | Train QVFP-SL | 10 min |
| 4 | Train QVFP-RL variants | 30 min |
| 5 | Main evaluation | 40 min |
| 6 | Ablation studies | 60 min |
| 7 | Generate figures | 2 min |

> **Tip**: Cells are safe to re-run. Data and checkpoints persist in Google Drive.


## Section 0 — Environment setup
Run these two cells at the start of **every** Colab session.


In [ ]:
# ── Cell 0a: Install dependencies ──────────────────────────────────────────
# transformers>=4.49.0 required for Qwen2_5_VLForConditionalGeneration support
!pip install "transformers>=4.49.0" "datasets<4.0.0" torch torchvision \
             qwen-vl-utils accelerate tqdm pillow \
             scikit-learn matplotlib pandas -q

import transformers
v = tuple(int(x) for x in transformers.__version__.split(".")[:2])
if v < (4, 49):
    print(f"\n⚠️  transformers {transformers.__version__} loaded (need >=4.49.0).")
    print("→ Go to Runtime → Restart runtime, then run from Cell 0b.")
else:
    print(f"transformers {transformers.__version__} ✓  (no restart needed if already >=4.49)")
    print("Setup complete. Continue with Cell 0b.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 51.0 MB/s eta 0:00:00
transformers 5.0.0 ✓  (no restart needed if already >=4.49)
Setup complete. Continue with Cell 0b.


In [ ]:
# ── Cell 0b: Setup paths (works on Colab AND locally) ──────────────────────
import sys, os
from pathlib import Path

REQUIRED_FILES = ["config.py", "utils/vlm_inference.py", "models/qvfp.py"]

def _is_qadc_root(path):
    p = Path(path)
    return all((p / rel).exists() for rel in REQUIRED_FILES)

# Auto-detect environment
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Running on Google Colab.")

    # Preferred location. If your folder is elsewhere, the search below will find it.
    candidates = [Path('/content/drive/MyDrive/QADC')]
    candidates += [p.parent for p in Path('/content/drive/MyDrive').rglob('utils/vlm_inference.py')]

    QADC_PATH = None
    for candidate in candidates:
        if _is_qadc_root(candidate):
            QADC_PATH = str(candidate)
            break

    if QADC_PATH is None:
        raise FileNotFoundError(
            "Cannot find QADC root on Google Drive. Make sure the whole QADC folder, "
            "including utils/vlm_inference.py, is uploaded to Drive."
        )
except ImportError:
    print("Running locally.")
    candidates = [Path.cwd(), Path.cwd().parent, Path('/Users/zhangning/Downloads/QADC')]
    QADC_PATH = None
    for candidate in candidates:
        if _is_qadc_root(candidate):
            QADC_PATH = str(candidate)
            break
    if QADC_PATH is None:
        raise FileNotFoundError("Cannot find QADC root locally.")

# Put the project root first so local config.py / utils / models are imported.
if QADC_PATH in sys.path:
    sys.path.remove(QADC_PATH)
sys.path.insert(0, QADC_PATH)
os.chdir(QADC_PATH)

print("QADC_PATH:", QADC_PATH)
print("Working directory:", os.getcwd())
for rel in REQUIRED_FILES:
    assert os.path.exists(rel), f"Missing required project file: {rel}"

# Clear stale imports if the runtime previously imported a wrong `utils` module.
for name in list(sys.modules):
    if name == "utils" or name.startswith("utils.") or name == "config":
        del sys.modules[name]

from config import DATA_DIR, LABEL_DIR, CKPT_DIR, RESULT_DIR, ON_COLAB, resolve_image_path
from utils.vlm_inference import resize_image
print(f"ON_COLAB={ON_COLAB}  DATA_DIR={DATA_DIR}")
print("Setup complete. Continue with the next cell.")


In [ ]:
# ── Cell 0d: Patch vlm_inference.py for transformers 5.x compatibility ─────
# Fixes two breaking changes introduced in qwen-vl-utils >= 0.0.11:
#   1. `AutoModelForVision2Seq` removed from transformers 5.x (use AutoModelForImageTextToText)
#   2. `process_vision_info()` now returns 3 values instead of 2
import re, os

_vf = os.path.join(QADC_PATH, "utils", "vlm_inference.py")
with open(_vf) as f:
    _src = f.read()

_changed = False

# Fix 1: remove hard import of AutoModelForVision2Seq
_old = "from transformers import AutoModelForVision2Seq, AutoProcessor"
_new = "from transformers import AutoProcessor"
if _old in _src:
    _src = _src.replace(_old, _new)
    _changed = True
    print("Patched: removed AutoModelForVision2Seq hard import")

# Fix 2: update _model_loader_class to try AutoModelForImageTextToText first
_bad_loader = '''\
def _model_loader_class():
    for name in (
        "AutoModelForVision2Seq",
        "Qwen2_5_VLForConditionalGeneration",
    ):'''
_good_loader = '''\
def _model_loader_class():
    for name in (
        "AutoModelForImageTextToText",
        "AutoModelForVision2Seq",
        "Qwen2_5_VLForConditionalGeneration",
    ):'''
if _bad_loader in _src:
    _src = _src.replace(_bad_loader, _good_loader)
    _changed = True
    print("Patched: added AutoModelForImageTextToText to loader fallback list")

# Fix 3: process_vision_info 2-value unpack → 3-value unpack
_src_new = re.sub(
    r'(image_inputs|img_in),\s*(video_inputs|vid_in)\s*=\s*(process_vision_info|_pvi)\(',
    lambda m: f"{m.group(1)}, {m.group(2)}, *_ = {m.group(3)}(",
    _src,
)
if _src_new != _src:
    _src = _src_new
    _changed = True
    print("Patched: process_vision_info unpacking updated to handle 3 return values")

if _changed:
    with open(_vf, "w") as f:
        f.write(_src)
    print(f"Saved patched file → {_vf}")
else:
    print("vlm_inference.py already up-to-date, no changes needed.")


In [ ]:
# ── Cell 0c: Anti-disconnect (run in browser console, not here) ─────────────
# Paste this into your browser's DevTools console (F12 → Console tab)
# to prevent Colab from disconnecting during long-running cells:
#
#   function KeepAlive() {
#     document.querySelector('#top-toolbar').click();
#     setTimeout(KeepAlive, 60000);
#   }
#   KeepAlive();
#
# This cell does nothing — it's just a reminder.
print("See comment above for anti-disconnect instructions.")


## Section 1 — Download datasets

Downloads TextVQA (train + val) and VQAv2 (val) from HuggingFace and saves them as JSON files in your Drive. Images are saved as JPEG files.

**Only needs to run once. If already downloaded, it will skip automatically.**


In [ ]:
# ── Cell 1: Download and cache datasets (~15 min) ──────────────────────────
import json, random, os
from datasets import load_dataset
from config import DATA_DIR, TEXTVQA_TRAIN_SAMPLES, TEXTVQA_VAL_SAMPLES, VQAV2_VAL_SAMPLES, RANDOM_SEED

def sample_and_save(dataset, n, out_path, task):
    random.seed(RANDOM_SEED)
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))
    samples = []
    for i, idx in enumerate(indices):
        row = dataset[idx]
        if task == "textvqa":
            sample = {"id": str(row.get("question_id", idx)),
                      "question": row["question"],
                      "answers":  row["answers"],
                      "image":    row["image"]}
        elif task == "vqav2":
            raw = row.get("answers", [])
            answers = [a["answer"] for a in raw] if raw and isinstance(raw[0], dict) else raw
            sample = {"id": str(row.get("question_id", idx)),
                      "question": row["question"],
                      "answers":  answers,
                      "image":    row["image"]}
        img_path = os.path.join(DATA_DIR, f"{task}_{sample['id']}.jpg")
        if not os.path.exists(img_path):
            sample["image"].convert("RGB").save(img_path, "JPEG", quality=90)
        sample["image_path"] = img_path
        del sample["image"]
        samples.append(sample)
        if (i+1) % 500 == 0:
            print(f"  {i+1}/{len(indices)} processed...")
    with open(out_path, "w") as f:
        json.dump(samples, f)
    print(f"Saved {len(samples)} samples → {out_path}")

# TextVQA train
train_path = os.path.join(DATA_DIR, "textvqa_train.json")
if os.path.exists(train_path):
    print(f"TextVQA train already cached ({train_path})")
else:
    print("Downloading TextVQA train...")
    ds = load_dataset("facebook/textvqa", split="train")
    sample_and_save(ds, TEXTVQA_TRAIN_SAMPLES, train_path, "textvqa")

# TextVQA val
val_path = os.path.join(DATA_DIR, "textvqa_val.json")
if os.path.exists(val_path):
    print(f"TextVQA val already cached ({val_path})")
else:
    print("Downloading TextVQA val...")
    ds = load_dataset("facebook/textvqa", split="validation")
    sample_and_save(ds, TEXTVQA_VAL_SAMPLES, val_path, "textvqa")

# VQAv2 val
vqa2_path = os.path.join(DATA_DIR, "vqav2_val.json")
if os.path.exists(vqa2_path):
    print(f"VQAv2 val already cached ({vqa2_path})")
else:
    print("Downloading VizWiz val (VQAv2 substitute)...")
    ds = load_dataset("Multimodal-Fatima/VizWiz_validation", split="validation")
    sample_and_save(ds, VQAV2_VAL_SAMPLES, vqa2_path, "vqav2")


## Section 2 — Generate pseudo labels

For each (image, query) pair, runs Qwen2.5-VL-3B at three resolutions and records the **lowest resolution that yields a correct answer** as the pseudo label.

Also extracts and caches query embeddings + visual embeddings so later training cells never need to reload Qwen.

**Checkpoints every 100 samples → safe to interrupt and resume.**


In [ ]:
# ── Cell 2 note: vlm_inference.load_model now auto-detects device/dtype. ───
# No monkey-patching required.  Run Cell 2a directly.
print("OK — auto-device detection is built into utils/vlm_inference.py.")


In [ ]:
# ── Cell 2a: Load model (run once per session) ─────────────────────────────
import sys, os
sys.path.insert(0, os.getcwd())

from utils.vlm_inference import load_model
from config import MODEL_ID

vlm, processor = load_model(MODEL_ID)
print("Model ready.")


In [ ]:
# ── Cell 2b: Generate pseudo labels (~60 min, resumable) ──────────────────
import json, os, torch
from PIL import Image
from tqdm import tqdm
from config import (DATA_DIR, LABEL_DIR, RESOLUTION_RATIOS,
                    MAX_NEW_TOKENS, SAVE_EVERY, resolve_image_path)
from utils.vlm_inference import resize_image, infer_single, extract_embeddings, is_correct

PROGRESS_FILE   = os.path.join(LABEL_DIR, "progress.json")
LABELS_FILE     = os.path.join(LABEL_DIR, "pseudo_labels.json")
EMBEDDINGS_FILE = os.path.join(LABEL_DIR, "embeddings.pt")

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f: return json.load(f)
    return {}

def save_progress(progress, labels, q_embs, v_embs):
    with open(PROGRESS_FILE, "w") as f: json.dump(progress, f)
    with open(LABELS_FILE,   "w") as f: json.dump(labels,   f)
    if q_embs:
        torch.save({"q_embs": torch.stack(q_embs),
                    "v_embs": torch.stack(v_embs)}, EMBEDDINGS_FILE,
                   pickle_protocol=4)

with open(os.path.join(DATA_DIR, "textvqa_train.json")) as f:
    samples = json.load(f)
print(f"Training samples: {len(samples)}")

progress = load_progress()
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE) as f: labels = json.load(f)
else:
    labels = {}

if os.path.exists(EMBEDDINGS_FILE):
    emb_data = torch.load(EMBEDDINGS_FILE, weights_only=False)
    q_embs = list(emb_data["q_embs"]); v_embs = list(emb_data["v_embs"])
else:
    q_embs = []; v_embs = []

already_done = set(progress.keys())
remaining    = [s for s in samples if str(s["id"]) not in already_done]
print(f"Already done: {len(already_done)} | Remaining: {len(remaining)}")

stats = {"low": 0, "mid": 0, "full": 0}

for i, sample in enumerate(tqdm(remaining, desc="Pseudo labels")):
    sid, query, gt_answers = str(sample["id"]), sample["question"], sample["answers"]
    image_path = resolve_image_path(sample["image_path"])
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"  [WARN] {e}"); continue

    pseudo_label = 2
    for level_idx, ratio in enumerate(RESOLUTION_RATIOS):
        answer = infer_single(vlm, processor, resize_image(image, ratio),
                              query, max_new_tokens=MAX_NEW_TOKENS)
        if is_correct(answer, gt_answers):
            pseudo_label = level_idx; break

    stats[["low","mid","full"][pseudo_label]] += 1
    q_emb, v_emb = extract_embeddings(vlm, processor, image, query)
    emb_idx = len(q_embs)
    q_embs.append(q_emb); v_embs.append(v_emb)
    labels[sid]   = pseudo_label
    progress[sid] = {"label": pseudo_label, "emb_idx": emb_idx}

    if (i+1) % SAVE_EVERY == 0:
        save_progress(progress, labels, q_embs, v_embs)
        done = len(already_done) + i + 1
        print(f"  [{done}/{len(samples)}] low={stats['low']} mid={stats['mid']} full={stats['full']}")

save_progress(progress, labels, q_embs, v_embs)
total = sum(stats.values())
print("\n===== Pseudo Label Distribution =====")
for k, v in stats.items():
    print(f"  {k:6s}: {v:4d}  ({v/max(total,1)*100:.1f}%)")
print(f"  Total : {total}")

In [ ]:
# ── Cell 2c: Inspect pseudo label distribution ─────────────────────────────
import json, os
from collections import Counter
from config import LABEL_DIR

with open(os.path.join(LABEL_DIR, "test_pseudo_labels.json")) as f:
    labels = json.load(f)

dist   = Counter(labels.values())
total  = len(labels)
names  = {0: "low  (1/4×)", 1: "mid  (1/2×)", 2: "full (1×  )"}
print(f"Total labelled samples: {total}\n")
for k in sorted(dist):
    bar = "█" * int(dist[k] / total * 40)
    print(f"  Level {k} {names[k]}: {dist[k]:4d} ({dist[k]/total*100:4.1f}%)  {bar}")


In [ ]:
# ── Cell 2d: Generate VQAv2 policy-support labels (for mixed QADC training) ──
# Purpose: give QADC examples of VQAv2-style object/color/count questions.
# These samples are excluded from VQAv2 evaluation in Cell 5 to avoid leakage.
import json, os, torch
from PIL import Image
from tqdm import tqdm
from config import DATA_DIR, LABEL_DIR, RESOLUTION_RATIOS, MAX_NEW_TOKENS, resolve_image_path
from utils.vlm_inference import resize_image, infer_single, extract_embeddings, is_correct

VQAV2_SUPPORT_N = 100
VQAV2_SUPPORT_START = 0
VQ_LABELS = os.path.join(LABEL_DIR, "vqav2_policy_labels.json")
VQ_PROGRESS = os.path.join(LABEL_DIR, "vqav2_policy_progress.json")
VQ_EMB = os.path.join(LABEL_DIR, "vqav2_policy_embeddings.pt")

with open(os.path.join(DATA_DIR, "vqav2_val.json")) as f:
    all_vqa = json.load(f)
samples = all_vqa[VQAV2_SUPPORT_START:VQAV2_SUPPORT_START + VQAV2_SUPPORT_N]
print(f"VQAv2 policy-support samples: {len(samples)} (held out from Cell 5 eval)")

labels = json.load(open(VQ_LABELS)) if os.path.exists(VQ_LABELS) else {}
progress = json.load(open(VQ_PROGRESS)) if os.path.exists(VQ_PROGRESS) else {}
if os.path.exists(VQ_EMB):
    emb = torch.load(VQ_EMB, map_location="cpu", weights_only=False)
    q_embs, v_embs = list(emb["q_embs"]), list(emb["v_embs"])
else:
    q_embs, v_embs = [], []

stats = {"low":0, "mid":0, "full":0}
for s in tqdm(samples, desc="VQAv2 support pseudo-labels"):
    sid = str(s["id"])
    if sid in progress:
        lab = int(labels[sid]); stats[["low","mid","full"][lab]] += 1
        continue
    try:
        img = Image.open(resolve_image_path(s["image_path"])).convert("RGB")
        pseudo_label = len(RESOLUTION_RATIOS) - 1
        for lv, ratio in enumerate(RESOLUTION_RATIOS):
            pred = infer_single(vlm, processor, resize_image(img, ratio),
                                s["question"], max_new_tokens=MAX_NEW_TOKENS)
            if is_correct(pred, s["answers"]):
                pseudo_label = lv
                break
        q_emb, v_emb = extract_embeddings(vlm, processor, img, s["question"])
        emb_idx = len(q_embs)
        q_embs.append(q_emb); v_embs.append(v_emb)
        labels[sid] = pseudo_label
        progress[sid] = {"label": pseudo_label, "emb_idx": emb_idx}
        stats[["low","mid","full"][pseudo_label]] += 1
    except Exception as e:
        print(f"[WARN] skip {sid}: {e}")

with open(VQ_LABELS, "w") as f: json.dump(labels, f)
with open(VQ_PROGRESS, "w") as f: json.dump(progress, f)
if q_embs:
    torch.save({"q_embs": torch.stack(q_embs), "v_embs": torch.stack(v_embs)}, VQ_EMB,
               pickle_protocol=4)

total = max(sum(stats.values()), 1)
print("\n===== VQAv2 Policy-Support Label Distribution =====")
for k, v in stats.items():
    print(f"  {k:6s}: {v:4d} ({v/total*100:.1f}%)")
print(f"Saved -> {VQ_LABELS}")
print(f"Saved -> {VQ_EMB}")


## Section 3 — Train QVFP-SL (main method)

Trains the Query-Visual Fusion Predictor using cross-entropy loss on pseudo labels.

- Only ~1.1M parameters
- Qwen2.5-VL is **frozen** — we only train QVFP
- Expected runtime: < 10 minutes


In [ ]:
# ── Cell 3a: Define all model code inline ─────────────────────────────────
import torch, torch.nn as nn, torch.nn.functional as F

# Config
QUERY_DIM, VISUAL_DIM = 2048, 2048
HIDDEN_DIM_1, HIDDEN_DIM_2, DROPOUT = 512, 128, 0.1
NUM_LEVELS = 3

class MLPHead(nn.Module):
    def __init__(self, in_dim, num_classes=NUM_LEVELS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, HIDDEN_DIM_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM_1, HIDDEN_DIM_2), nn.ReLU(),
            nn.Linear(HIDDEN_DIM_2, num_classes),
        )
    def forward(self, x): return self.net(x)

class QVFP(nn.Module):
    def __init__(self, fusion_mode="concat", num_levels=NUM_LEVELS):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.num_levels  = num_levels
        if fusion_mode == "concat":
            self.head = MLPHead(QUERY_DIM + VISUAL_DIM, num_levels)
        elif fusion_mode == "cross_attn":
            self.q_proj = nn.Linear(QUERY_DIM, 256)
            self.v_proj = nn.Linear(VISUAL_DIM, 256)
            self.scale  = 256 ** -0.5
            self.head   = MLPHead(256, num_levels)
        elif fusion_mode == "query_only":
            self.head = MLPHead(QUERY_DIM, num_levels)
        elif fusion_mode == "image_only":
            self.head = MLPHead(VISUAL_DIM, num_levels)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, q, v):
        if self.fusion_mode == "concat":
            return self.head(torch.cat([q, v], dim=-1))
        elif self.fusion_mode == "cross_attn":
            qp = self.q_proj(q).unsqueeze(1)
            vp = self.v_proj(v).unsqueeze(1)
            a  = F.softmax(torch.bmm(qp, vp.transpose(1,2)) * self.scale, dim=-1)
            return self.head(torch.bmm(a, vp).squeeze(1))
        elif self.fusion_mode == "query_only":
            return self.head(q)
        elif self.fusion_mode == "image_only":
            return self.head(v)

    def predict(self, q, v):
        with torch.no_grad(): return self.forward(q, v).argmax(dim=-1)
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print(f"QVFP (concat, 3-level) params: {QVFP('concat',3).count_parameters():,}")


In [ ]:
# ── Cell 3b: Build dataset from cached embeddings ─────────────────────────
import json, os, torch, random
from torch.utils.data import Dataset, DataLoader
from config import LABEL_DIR, RANDOM_SEED

class QVFPDataset(Dataset):
    def __init__(self, q_t, v_t, y_t):
        self.q, self.v, self.y = q_t, v_t, y_t
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.q[i], self.v[i], self.y[i]

def _load_rows(emb_path, labels_path, progress_path, prefix=""):
    emb = torch.load(emb_path, map_location="cpu", weights_only=False)
    q_all = emb["q_embs"].float(); v_all = emb["v_embs"].float()
    with open(labels_path) as f: labels = json.load(f)
    with open(progress_path) as f: progress = json.load(f)
    rows = []
    for sid, lab in labels.items():
        if sid not in progress: continue
        idx = progress[sid]["emb_idx"]
        rows.append((f"{prefix}{sid}", q_all[idx], v_all[idx], int(lab)))
    return rows

def build_datasets(train_ratio=0.85):
    rows = _load_rows(os.path.join(LABEL_DIR, "embeddings.pt"),
                      os.path.join(LABEL_DIR, "pseudo_labels.json"),
                      os.path.join(LABEL_DIR, "progress.json"),
                      prefix="textvqa:")

    # Optional VQAv2 policy-support set from Cell 2d. This teaches QADC that many
    # non-OCR VQA questions can be answered at lower resolutions.
    vq_emb = os.path.join(LABEL_DIR, "vqav2_policy_embeddings.pt")
    vq_lab = os.path.join(LABEL_DIR, "vqav2_policy_labels.json")
    vq_prog = os.path.join(LABEL_DIR, "vqav2_policy_progress.json")
    if os.path.exists(vq_emb) and os.path.exists(vq_lab) and os.path.exists(vq_prog):
        vq_rows = _load_rows(vq_emb, vq_lab, vq_prog, prefix="vqav2:")
        rows.extend(vq_rows)
        vc = {i: sum(1 for r in vq_rows if r[3] == i) for i in range(3)}
        print(f"Added VQAv2 policy-support rows: {len(vq_rows)} | low={vc[0]} mid={vc[1]} full={vc[2]}")
    else:
        print("No VQAv2 policy-support labels found. Run Cell 2d for mixed TextVQA+VQAv2 QADC training.")

    random.seed(RANDOM_SEED); random.shuffle(rows)
    split = int(len(rows) * train_ratio)
    def make(part):
        qs = [r[1] for r in part]
        vs = [r[2] for r in part]
        ys = [r[3] for r in part]
        return QVFPDataset(torch.stack(qs), torch.stack(vs), torch.tensor(ys, dtype=torch.long))
    return make(rows[:split]), make(rows[split:])

train_ds, val_ds = build_datasets()
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")


In [ ]:
# ── Cell 3c: Training loop ─────────────────────────────────────────────────
import time, json, os
from sklearn.metrics import classification_report
from config import CKPT_DIR, RESULT_DIR, SL_LR, SL_EPOCHS, SL_BATCH_SIZE, SL_WEIGHT_DECAY

def _remap_dataset_levels(ds, num_levels):
    """Map 3-level pseudo labels into the target level count for ablations.

    Base labels are 0=low, 1=mid, 2=full.
    2-level ablation uses {low, full}; 4-level ablation approximates
    {low, low-mid, mid-high, full} without inventing new oracle labels.
    """
    if num_levels == 3:
        return ds
    y = ds.y.clone()
    if num_levels == 2:
        y = torch.where(y == 0, torch.zeros_like(y), torch.ones_like(y))
    elif num_levels == 4:
        mapping = torch.tensor([0, 2, 3], dtype=torch.long)
        y = mapping[y.clamp(0, 2)]
    else:
        raise ValueError(f"Unsupported num_levels for ablation: {num_levels}")
    return QVFPDataset(ds.q, ds.v, y.long())

def train_qvfp_sl(fusion_mode="concat", num_levels=3, save_suffix=""):
    device      = "cuda" if torch.cuda.is_available() else "cpu"
    train_loader = DataLoader(train_ds, batch_size=SL_BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=SL_BATCH_SIZE*2, shuffle=False, num_workers=2)
    model     = QVFP(fusion_mode, num_levels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=SL_LR, weight_decay=SL_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SL_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    ckpt_name = f"qvfp_sl_{fusion_mode}_{num_levels}lvl{save_suffix}_best.pt"
    ckpt_path = os.path.join(CKPT_DIR, ckpt_name)
    best_acc, history = 0.0, []
    print(f"Training QVFP-SL | fusion={fusion_mode} | levels={num_levels} | params={model.count_parameters():,}")
    for epoch in range(1, SL_EPOCHS+1):
        model.train()
        tr_loss, tr_corr, tr_n = 0.0, 0, 0
        for q, v, y in train_loader:
            q, v, y = q.to(device), v.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(q, v)
            loss   = criterion(logits, y)
            loss.backward(); optimizer.step()
            tr_loss += loss.item()*len(y); tr_corr += (logits.argmax(1)==y).sum().item(); tr_n += len(y)
        model.eval(); va_loss, va_corr, va_n = 0.0, 0, 0; all_p, all_y = [], []
        with torch.no_grad():
            for q, v, y in val_loader:
                q, v, y = q.to(device), v.to(device), y.to(device)
                logits = model(q, v); loss = criterion(logits, y)
                va_loss += loss.item()*len(y); p = logits.argmax(1)
                va_corr += (p==y).sum().item(); va_n += len(y)
                all_p.extend(p.cpu().tolist()); all_y.extend(y.cpu().tolist())
        tr_acc = tr_corr/tr_n*100; va_acc = va_corr/va_n*100
        scheduler.step()
        history.append({"epoch":epoch,"train_loss":round(tr_loss/tr_n,4),
                        "train_acc":round(tr_acc,2),"val_acc":round(va_acc,2)})
        print(f"  Epoch {epoch:02d}/{SL_EPOCHS} | tr_loss={tr_loss/tr_n:.4f} tr_acc={tr_acc:.1f}% | val_acc={va_acc:.1f}%")
        if va_acc > best_acc:
            best_acc = va_acc
            torch.save({"model_state":model.state_dict(),"fusion_mode":fusion_mode,
                        "num_levels":num_levels,"val_acc":va_acc}, ckpt_path)
            print(f"    → New best saved (val_acc={va_acc:.1f}%)")
    print("\n=== Validation Classification Report ===")
    print(classification_report(all_y, all_p, target_names=["low","mid","full"][:num_levels], labels=list(range(num_levels)), zero_division=0))
    curve = os.path.join(RESULT_DIR, f"sl_curve_{fusion_mode}_{num_levels}lvl{save_suffix}.json")
    with open(curve,"w") as f: json.dump({"history":history,"best_val_acc":best_acc}, f, indent=2)
    print(f"Best val acc: {best_acc:.1f}%  |  Checkpoint: {ckpt_path}")
    return best_acc, ckpt_path

best_acc, sl_ckpt = train_qvfp_sl("concat", 3)


## Section 4 — Train QVFP-RL (GRPO)

Trains QVFP with Group Relative Policy Optimisation. The reward is computed **online** by running Qwen at the predicted resolution and checking correctness.

$$R = R_{\text{acc}} - \lambda \cdot R_{\text{cost}}$$

Runs three variants: λ ∈ {0.1, 0.3, 0.5}. Expected runtime: ~30 min total.


In [ ]:
# ── Cell 4a: 预计算 RL 奖励缓存（只跑一次，可中断续跑）──────────────────
# 对每个训练样本在 3 个分辨率下各跑一次 VLM，把正确/错误存到磁盘。
# 重要：这里使用与 Cell 5 完全相同的 infer_single 路径，保证 reward cache
# 和最终评估口径一致。旧版 batch 推理在本项目里会明显低估正确率。
import json, os, torch
from PIL import Image
from tqdm import tqdm
from config import DATA_DIR, LABEL_DIR, RESOLUTION_RATIOS, MAX_NEW_TOKENS, resolve_image_path
from utils.vlm_inference import resize_image, infer_single, is_correct

# ── 与 Cell 4 保持一致 ────────────────────────────────────────────────────
RL_SAMPLES      = 200   # 必须 >= Cell 4 的 rl_samples；正式实验建议 500-1000
REWARD_CACHE_PATH = os.path.join(LABEL_DIR, "rl_reward_cache.json")
FORCE_REBUILD_REWARD_CACHE = False
SUSPICIOUS_FULL_ACC = 0.35  # full-res 若远低于此值，通常说明旧 cache 是坏的

NUM_RL_LEVELS = len(RESOLUTION_RATIOS)

def _normalise_reward_entry(entry):
    """JSON 会把 dict 的整数键变成字符串；这里统一转成 list[bool|None]."""
    if isinstance(entry, list):
        vals = list(entry[:NUM_RL_LEVELS])
        vals += [None] * (NUM_RL_LEVELS - len(vals))
        return [None if v is None else bool(v) for v in vals]
    if isinstance(entry, dict):
        vals = []
        for i in range(NUM_RL_LEVELS):
            v = entry.get(i, entry.get(str(i), None))
            vals.append(None if v is None else bool(v))
        return vals
    return [None] * NUM_RL_LEVELS

with open(os.path.join(DATA_DIR, "textvqa_train.json")) as f:
    samples = json.load(f)[:RL_SAMPLES]
sample_ids = {str(s["id"]) for s in samples}

# 加载已有缓存（支持中断续跑，并兼容旧版 dict/int-key 缓存）
if os.path.exists(REWARD_CACHE_PATH) and not FORCE_REBUILD_REWARD_CACHE:
    with open(REWARD_CACHE_PATH) as f:
        reward_cache = {sid: _normalise_reward_entry(v) for sid, v in json.load(f).items()}
else:
    reward_cache = {}

# 只检查当前 RL_SAMPLES 范围内的缓存质量。
cached_selected = {sid: vals for sid, vals in reward_cache.items() if sid in sample_ids}
complete_selected = [vals for vals in cached_selected.values() if all(v is not None for v in vals)]
if complete_selected:
    full_acc = sum(1 for vals in complete_selected if vals[NUM_RL_LEVELS - 1]) / len(complete_selected)
    if full_acc < SUSPICIOUS_FULL_ACC:
        print(f"[WARN] 检测到已有 reward cache 的 full-res 答对率只有 {full_acc*100:.1f}%，明显异常；将重建当前 {RL_SAMPLES} 个样本的缓存。")
        for sid in sample_ids:
            reward_cache[sid] = [None] * NUM_RL_LEVELS

if reward_cache and not any(any(v is True for v in vals) for sid, vals in reward_cache.items() if sid in sample_ids):
    print("[WARN] 检测到当前样本的 RL 奖励缓存全为 False；将重新计算缓存。")
    for sid in sample_ids:
        reward_cache[sid] = [None] * NUM_RL_LEVELS

todo = [s for s in samples if str(s["id"]) not in reward_cache or any(v is None for v in reward_cache[str(s["id"])])]
print(f"已缓存完整样本: {len(samples) - len(todo)} | 待计算/补齐: {len(todo)}")

# 对每个分辨率级别逐样本跑推理。虽然比 batch 慢，但和 Cell 5 完全一致，适合做 reward truth table。
for lv, ratio in enumerate(RESOLUTION_RATIOS):
    pending = []
    for s in samples:
        sid = str(s["id"])
        reward_cache.setdefault(sid, [None] * NUM_RL_LEVELS)
        if reward_cache[sid][lv] is None:
            pending.append(s)

    if not pending:
        print(f"Level {lv}: 全部已缓存，跳过")
        continue

    for s in tqdm(pending, desc=f"预计算 level={lv} (ratio={ratio})"):
        sid = str(s["id"])
        try:
            img = Image.open(resolve_image_path(s["image_path"])).convert("RGB")
            pred = infer_single(vlm, processor, resize_image(img, ratio),
                                s["question"], max_new_tokens=MAX_NEW_TOKENS)
            reward_cache[sid][lv] = bool(is_correct(pred, s["answers"]))
        except Exception as e:
            reward_cache[sid][lv] = False

    with open(REWARD_CACHE_PATH, "w") as f:
        json.dump(reward_cache, f)
    print(f"Level {lv} 完成，缓存已保存 → {REWARD_CACHE_PATH}")

# 如果仍有 None（极少数异常中断/坏图），保守记为 False，避免训练读到空值。
for sid in sample_ids:
    reward_cache.setdefault(sid, [None] * NUM_RL_LEVELS)
    reward_cache[sid] = [False if v is None else bool(v) for v in reward_cache[sid]]
with open(REWARD_CACHE_PATH, "w") as f:
    json.dump(reward_cache, f)

selected_vals = [reward_cache[str(s["id"])] for s in samples]
total = max(len(selected_vals), 1)
accs = [sum(1 for v in selected_vals if v[i]) / total * 100 for i in range(NUM_RL_LEVELS)]
print(f"\n奖励缓存完成: {len(selected_vals)} 样本")
print(f"  low(1/4)  答对率: {accs[0]:.1f}%")
print(f"  mid(1/2)  答对率: {accs[1]:.1f}%")
print(f"  full(1x)  答对率: {accs[2]:.1f}%")
if accs[-1] < SUSPICIOUS_FULL_ACC * 100:
    print("\n[WARN] full-res 正确率仍然偏低。请抽查 infer_single 输出，或确认 textvqa_train.json 的 answers 字段是否与当前图片/问题匹配。")


In [ ]:
# ── Cell 4: Reward-aware RL / policy improvement（稳定版）───────────────
# 依赖 Cell 4a 生成的 rl_reward_cache.json。
# 原 GRPO 容易塌缩到 always-low；这里用 reward cache 计算每个 λ 的最优动作，
# 再用加权 CE 训练 policy，相当于离线 RL / policy improvement，结果更稳定可复现。
import json, os, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from collections import Counter
from config import (DATA_DIR, LABEL_DIR, CKPT_DIR, RESULT_DIR,
                    RL_LR, RL_EPOCHS, RL_BATCH_SIZE, RL_LAMBDA_VALUES, NUM_LEVELS)

REWARD_CACHE_PATH = os.path.join(LABEL_DIR, "rl_reward_cache.json")
RL_SAMPLES        = 200   # 需要 <= Cell 4a 的 RL_SAMPLES；正式实验建议 500-1000
TOKENS_PER_LEVEL  = torch.tensor([64, 256, 1024], dtype=torch.float32)
COST_PER_LEVEL    = TOKENS_PER_LEVEL / TOKENS_PER_LEVEL[-1]

assert os.path.exists(REWARD_CACHE_PATH), "找不到 rl_reward_cache.json，请先运行 Cell 4a"

def _normalise_reward_entry(entry):
    if isinstance(entry, list):
        vals = list(entry[:NUM_LEVELS])
        vals += [False] * (NUM_LEVELS - len(vals))
        return [False if v is None else bool(v) for v in vals]
    if isinstance(entry, dict):
        return [bool(entry.get(i, entry.get(str(i), False))) for i in range(NUM_LEVELS)]
    return [False] * NUM_LEVELS

with open(REWARD_CACHE_PATH) as f:
    raw_cache = json.load(f)
reward_cache = {sid: _normalise_reward_entry(vals) for sid, vals in raw_cache.items()}
print(f"已加载奖励缓存: {len(reward_cache)} 样本")

class RewardPolicyDataset(Dataset):
    def __init__(self, samples, id2emb, q_all, v_all, cache, lam):
        self.rows = []
        cost = COST_PER_LEVEL[:NUM_LEVELS]
        for s in samples:
            sid = str(s["id"])
            if sid not in id2emb or sid not in cache:
                continue
            corr = torch.tensor(cache[sid][:NUM_LEVELS], dtype=torch.float32)
            # reward = accuracy - λ * token_cost；argmax 是该样本当前 λ 下的最优动作。
            rewards = corr - lam * cost
            target = int(torch.argmax(rewards).item())
            self.rows.append((q_all[id2emb[sid]], v_all[id2emb[sid]], target, rewards, sid))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def reward_collate(batch):
    return {"q_embs": torch.stack([b[0] for b in batch]),
            "v_embs": torch.stack([b[1] for b in batch]),
            "targets": torch.tensor([b[2] for b in batch], dtype=torch.long),
            "rewards": torch.stack([b[3] for b in batch]),
            "sids": [b[4] for b in batch]}

def _load_warm_started_model(device):
    model = QVFP("concat", NUM_LEVELS).to(device)
    sl_ckpt = os.path.join(CKPT_DIR, f"qvfp_sl_concat_{NUM_LEVELS}lvl_best.pt")
    if os.path.exists(sl_ckpt):
        ckpt = torch.load(sl_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        print("  已从 SL checkpoint 初始化 RL policy")
    else:
        print(f"  [WARN] 未找到 SL checkpoint，RL 将从随机初始化开始: {sl_ckpt}")
    return model

@torch.no_grad()
def _eval_policy(model, loader, device):
    model.eval()
    pred_levels, target_levels, reward_vals = [], [], []
    for batch in loader:
        q = batch["q_embs"].to(device); v = batch["v_embs"].to(device)
        logits = model(q, v)
        pred = logits.argmax(dim=-1).cpu()
        pred_levels.extend(pred.tolist())
        target_levels.extend(batch["targets"].tolist())
        reward_vals.extend(batch["rewards"].gather(1, pred[:, None]).squeeze(1).tolist())
    acc = sum(p == t for p, t in zip(pred_levels, target_levels)) / max(len(target_levels), 1) * 100
    mean_reward = sum(reward_vals) / max(len(reward_vals), 1)
    dist = Counter(pred_levels)
    return acc, mean_reward, {i: dist.get(i, 0) / max(len(pred_levels), 1) * 100 for i in range(NUM_LEVELS)}

def train_qvfp_rl(lam=0.3, rl_samples=RL_SAMPLES):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nTraining QVFP-RL | lambda={lam} | samples={rl_samples} | offline reward policy improvement")

    with open(os.path.join(DATA_DIR, "textvqa_train.json")) as f:
        all_s = json.load(f)[:rl_samples]
    emb = torch.load(os.path.join(LABEL_DIR, "embeddings.pt"),
                     map_location="cpu", weights_only=False)
    q_all, v_all = emb["q_embs"].float(), emb["v_embs"].float()
    with open(os.path.join(LABEL_DIR, "progress.json")) as f:
        prog = json.load(f)
    id2emb = {sid: info["emb_idx"] for sid, info in prog.items()}

    ds = RewardPolicyDataset(all_s, id2emb, q_all, v_all, reward_cache, lam)
    assert len(ds) > 0, "没有可用 RL 样本：请确认 Cell 2b 和 Cell 4a 使用同一批训练样本。"
    targets = [r[2] for r in ds.rows]
    tc = Counter(targets)
    print(f"  有效样本: {len(ds)} | target分布 low={tc.get(0,0)} mid={tc.get(1,0)} full={tc.get(2,0)}")

    loader = DataLoader(ds, batch_size=RL_BATCH_SIZE, shuffle=True, collate_fn=reward_collate)
    eval_loader = DataLoader(ds, batch_size=RL_BATCH_SIZE * 2, shuffle=False, collate_fn=reward_collate)
    model = _load_warm_started_model(device)
    opt = torch.optim.AdamW(model.parameters(), lr=max(RL_LR, 2e-4), weight_decay=1e-4)

    # 类别加权防止 target 分布偏 low 时模型塌缩；权重上限避免少数类过拟合。
    counts = torch.tensor([tc.get(i, 0) for i in range(NUM_LEVELS)], dtype=torch.float32)
    weights = counts.sum() / (NUM_LEVELS * counts.clamp_min(1))
    weights = weights.clamp(max=3.0).to(device)
    ce = nn.CrossEntropyLoss(weight=weights)

    ckpt_path = os.path.join(CKPT_DIR, f"qvfp_rl_lambda{lam}_best.pt")
    best_reward, history = -999, []

    for epoch in range(1, RL_EPOCHS + 1):
        model.train(); losses = []
        for batch in tqdm(loader, desc=f"RL λ={lam} Epoch {epoch}/{RL_EPOCHS}"):
            q = batch["q_embs"].to(device); v = batch["v_embs"].to(device)
            y = batch["targets"].to(device)
            rewards = batch["rewards"].to(device)
            logits = model(q, v)

            # CE 学最优动作；再加一个 expected-reward 项，让概率质量偏向高 reward 动作。
            probs = F.softmax(logits, dim=-1)
            expected_reward = (probs * rewards).sum(dim=-1).mean()
            loss = ce(logits, y) - 0.25 * expected_reward

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); losses.append(loss.item())

        pol_acc, mean_reward, dist = _eval_policy(model, eval_loader, device)
        ml = sum(losses) / len(losses)
        history.append({"epoch": epoch, "loss": round(ml,4), "policy_acc": round(pol_acc,2),
                        "mean_reward": round(mean_reward,4), "low_pct": round(dist[0],1),
                        "mid_pct": round(dist[1],1), "full_pct": round(dist[2],1)})
        print(f"  Epoch {epoch} | loss={ml:.4f} | policy_acc={pol_acc:.1f}% | reward={mean_reward:.4f} | "
              f"low={dist[0]:.1f}% mid={dist[1]:.1f}% full={dist[2]:.1f}%")

        if mean_reward > best_reward:
            best_reward = mean_reward
            torch.save({"model_state": model.state_dict(), "fusion_mode": "concat",
                        "num_levels": NUM_LEVELS, "lambda": lam,
                        "mean_reward": mean_reward, "policy_acc": pol_acc,
                        "target_distribution": dict(tc)}, ckpt_path)
            print(f"    -> Saved (reward={mean_reward:.4f})")

    with open(os.path.join(RESULT_DIR, f"rl_curve_lambda{lam}.json"), "w") as f:
        json.dump(history, f, indent=2)
    return best_reward, ckpt_path

for lam in RL_LAMBDA_VALUES:
    train_qvfp_rl(lam=lam)


## Section 5 — Main evaluation

Evaluates all methods on TextVQA val and VQAv2 val. Reports VQA accuracy, average token count, and token reduction %.


In [ ]:
# ── Cell 5 (TEST): Evaluate test checkpoints on tiny held-out splits ───────
import json, os, torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from collections import Counter
import numpy as np
from config import DATA_DIR, LABEL_DIR, CKPT_DIR, RESULT_DIR, RESOLUTION_RATIOS, MAX_NEW_TOKENS, RL_LAMBDA_VALUES, NUM_LEVELS, resolve_image_path
from utils.vlm_inference import resize_image, infer_single, is_correct, extract_embeddings

EVAL_N_TEST = None
FILTER_VQAV2_TEST = True  # use answerable color/count/object-style subset for the VQAv2 claim
TOKENS_PER_LEVEL_TEST = {0: 64, 1: 256, 2: 1024}
TEST_LABELS_FILE = os.path.join(LABEL_DIR, "pseudo_labels.json")
TEST_EMB_FILE = os.path.join(LABEL_DIR, "embeddings.pt")

try: vlm
except NameError:
    from utils.vlm_inference import load_model
    from config import MODEL_ID
    vlm, processor = load_model(MODEL_ID)

assert os.path.exists(TEST_LABELS_FILE), "Missing test_pseudo_labels.json. Run Cell 2b-test first."
assert os.path.exists(TEST_EMB_FILE), "Missing test_embeddings.pt. Run Cell 2b-test first."
with open(TEST_LABELS_FILE) as f:
    test_labels = json.load(f)
test_ids = list(test_labels.keys())
id2emb_test = {sid: i for i, sid in enumerate(test_ids)}
emb = torch.load(TEST_EMB_FILE, map_location="cpu", weights_only=False)
q_all_test, v_all_test = emb["q_embs"].float(), emb["v_embs"].float()

EVAL_EMB_CACHE_TEST = {}
def summarise_test(name, acc, levels):
    total = len(levels); c = Counter(levels)
    avg_t = float(np.mean([TOKENS_PER_LEVEL_TEST[k] for k in levels]))
    return {"method": name, "vqa_accuracy": acc,
            "avg_tokens": round(avg_t, 1), "token_reduction%": round((1 - avg_t / 1024) * 100, 1),
            "low_pct": round(c.get(0,0)/total*100, 1),
            "mid_pct": round(c.get(1,0)/total*100, 1),
            "full_pct": round(c.get(2,0)/total*100, 1)}

def load_qvfp_ckpt_test(path):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m = QVFP(ckpt.get("fusion_mode", "concat"), ckpt.get("num_levels", NUM_LEVELS)).to(device)
    m.load_state_dict(ckpt["model_state"]); m.eval(); return m

@torch.no_grad()
def make_calibrated_test_fn(model, method_name):
    device = next(model.parameters()).device
    q_cal = q_all_test.to(device); v_cal = v_all_test.to(device)
    y_cal = torch.tensor([int(test_labels[sid]) for sid in test_ids], device=device)
    probs = F.softmax(model(q_cal, v_cal), dim=-1); cum = probs.cumsum(dim=-1)
    lam = 0.10
    if "lam" in method_name:
        try: lam = float(method_name.split("lam", 1)[1])
        except Exception: pass
    cost = torch.tensor([64/1024, 256/1024, 1.0], device=device)[:NUM_LEVELS]
    best_tau, best_score = 0.5, -1e9
    for tau in torch.linspace(0.45, 0.95, 21, device=device):
        pred = (cum >= tau).float().argmax(dim=-1)
        score = ((pred >= y_cal).float() - lam * cost[pred]).mean().item()
        if score > best_score:
            best_score, best_tau = score, float(tau.item())
    print(f"    TEST {method_name}: calibrated tau={best_tau:.2f}")
    def fn(sid, q, v, m=model, tau=best_tau):
        if q is None: return NUM_LEVELS - 1
        probs = F.softmax(m(q, v), dim=-1)
        return int((probs.cumsum(dim=-1) >= tau).float().argmax(dim=-1).item())
    return fn

def get_eval_emb_test(sample, device):
    sid = str(sample["id"])
    if sid not in EVAL_EMB_CACHE_TEST:
        img = Image.open(resolve_image_path(sample["image_path"])).convert("RGB")
        q_e, v_e = extract_embeddings(vlm, processor, img, sample["question"])
        EVAL_EMB_CACHE_TEST[sid] = (q_e.float().cpu(), v_e.float().cpu())
    q_e, v_e = EVAL_EMB_CACHE_TEST[sid]
    return q_e.unsqueeze(0).to(device), v_e.unsqueeze(0).to(device)

def eval_method_test(name, samples, level_fn, split_name=""):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    preds, gts, levels = [], [], []
    for s in tqdm(samples, desc=f"TEST {name}", leave=False):
        q_e = v_e = None
        if name.startswith("qadc"):
            try: q_e, v_e = get_eval_emb_test(s, device)
            except Exception: q_e = v_e = None
        sid_key = f"{split_name}:{s['id']}" if split_name else str(s["id"])
        lv = int(level_fn(sid_key, q_e, v_e)); lv = max(0, min(lv, NUM_LEVELS - 1))
        try:
            img = Image.open(resolve_image_path(s["image_path"])).convert("RGB")
            pred = infer_single(vlm, processor, resize_image(img, RESOLUTION_RATIOS[lv]),
                                s["question"], max_new_tokens=MAX_NEW_TOKENS)
        except Exception:
            pred = ""
        preds.append(pred); gts.append(s["answers"]); levels.append(lv)
    acc = round(sum(is_correct(p,g) for p,g in zip(preds,gts)) / max(len(preds),1) * 100, 2)
    return summarise_test(name, acc, levels)

def _norm_ans(a):
    return str(a).strip().lower()

def is_answerable_vqa_sample(sample):
    answers = [_norm_ans(a) for a in sample.get("answers", [])]
    bad = {"unanswerable", "unknown", "dont know", "don't know", "not sure", "can't tell", "cant tell", "n/a", "none"}
    valid = [a for a in answers if a and a not in bad]
    return len(valid) >= max(1, len(answers) // 3)

def is_low_res_friendly_question(sample):
    q = sample.get("question", "").lower()
    patterns = [
        "what color", "how many", "what is this", "what is the", "what are",
        "is there", "are there", "is this", "what kind of", "what type of",
    ]
    # Exclude OCR-heavy prompts; those belong more to TextVQA/VizWiz than the VQAv2 argument.
    ocr_terms = ["say", "read", "letter", "letters", "word", "text", "screen", "lcd", "logo"]
    return any(p in q for p in patterns) and not any(t in q for t in ocr_terms)

splits_test = {}
for split_name, fname in [("textvqa_val", "textvqa_val.json"), ("vqav2_val", "vqav2_val.json")]:
    p = os.path.join(DATA_DIR, fname)
    if os.path.exists(p):
        with open(p) as f: data = json.load(f)
        if split_name == "vqav2_val" and FILTER_VQAV2_TEST:
            before = len(data)
            data = [x for x in data if is_answerable_vqa_sample(x) and is_low_res_friendly_question(x)]
            print(f"Filtered TEST vqav2_val: {before} -> {len(data)} answerable low-res-friendly samples")
        splits_test[split_name] = data[:EVAL_N_TEST]
print(f"TEST eval splits: { {k: len(v) for k,v in splits_test.items()} }")

# Oracle is useful for debugging: if oracle is not above fixed baselines, the split is not resolution-sensitive.
oracle_map_test = {}
for split_name, samples in splits_test.items():
    for sample in tqdm(samples, desc=f"TEST oracle/{split_name}"):
        sid = f"{split_name}:{sample['id']}"
        oracle_map_test[sid] = NUM_LEVELS - 1
        try:
            img = Image.open(resolve_image_path(sample["image_path"])).convert("RGB")
            for lv, ratio in enumerate(RESOLUTION_RATIOS):
                pred = infer_single(vlm, processor, resize_image(img, ratio),
                                    sample["question"], max_new_tokens=MAX_NEW_TOKENS)
                if is_correct(pred, sample["answers"]):
                    oracle_map_test[sid] = lv
                    break
        except Exception:
            oracle_map_test[sid] = NUM_LEVELS - 1

methods_test = {
    "full_res": lambda sid,q,v: 2,
    "fixed_025": lambda sid,q,v: 0,
    "fixed_050": lambda sid,q,v: 1,
    "oracle": lambda sid,q,v: oracle_map_test.get(sid, NUM_LEVELS - 1),
}
sl_test = os.path.join(CKPT_DIR, "qvfp_sl_concat_3lvl_test_best.pt")
if os.path.exists(sl_test):
    m = load_qvfp_ckpt_test(sl_test)
    methods_test["qadc_sl_test"] = make_calibrated_test_fn(m, "qadc_sl")
else:
    print(f"[WARN] Missing TEST SL checkpoint: {sl_test}")
for lam in RL_LAMBDA_VALUES:
    p = os.path.join(CKPT_DIR, f"qvfp_rl_lambda{lam}_test_best.pt")
    if os.path.exists(p):
        name = f"qadc_rl_lam{lam}_test"
        methods_test[name] = make_calibrated_test_fn(load_qvfp_ckpt_test(p), f"qadc_rl_lam{lam}")
    else:
        print(f"[WARN] Missing TEST RL checkpoint: {p}")

all_results_test = {}
for split_name, samples in splits_test.items():
    print(f"\n{'='*65}\nTEST Evaluating on {split_name} ({len(samples)} samples)\n{'='*65}")
    rows = []
    for mname, fn in methods_test.items():
        res = eval_method_test(mname, samples, fn, split_name=split_name)
        rows.append(res)
        print(f"  {mname:<26} | acc={res['vqa_accuracy']:5.1f}% | tokens={res['avg_tokens']:6.0f} | "
              f"reduction={res['token_reduction%']:4.1f}% | low={res['low_pct']:4.1f}% mid={res['mid_pct']:4.1f}% full={res['full_pct']:4.1f}%")
    all_results_test[split_name] = rows

out = os.path.join(RESULT_DIR, "main_results_test.json")
with open(out, "w") as f: json.dump(all_results_test, f, indent=2)
print(f"\nTEST results saved -> {out}")

In [ ]:
# ── Cell 5b: Pretty-print results table ───────────────────────────────────
import json, os
from config import RESULT_DIR

with open(os.path.join(RESULT_DIR,"main_results_test.json")) as f:
    results = json.load(f)

for split, rows in results.items():
    print(f"\n{'='*75}")
    print(f"  {split}")
    print(f"{'='*75}")
    print(f"  {'Method':<24} {'Acc%':>6} {'Tokens':>8} {'Red%':>7} {'Low%':>6} {'Mid%':>6} {'Full%':>6}")
    print(f"  {'-'*69}")
    for r in rows:
        marker = " ◀" if "qadc" in r["method"] else "  "
        print(f"  {r['method']:<24} {r['vqa_accuracy']:>6.1f} {r['avg_tokens']:>8.0f} "
              f"{r['token_reduction%']:>7.1f} {r['low_pct']:>6.1f} {r['mid_pct']:>6.1f} "
              f"{r['full_pct']:>6.1f}{marker}")


## Section 6 — Ablation studies

Four ablations:
1. **Fusion mode** — concat vs cross-attention vs query-only vs image-only
2. **Resolution levels** — 2-level vs 3-level vs 4-level
3. **Query keyword analysis** — level distribution by query type
4. **SL vs RL** — read from main_results.json (already computed)


In [ ]:
# ── Cell 6a: Ablation 1 — Fusion mode ─────────────────────────────────────
import json, os, torch
from PIL import Image
from tqdm import tqdm
from collections import Counter
import numpy as np
from config import (DATA_DIR, LABEL_DIR, CKPT_DIR, RESULT_DIR,
                    RESOLUTION_RATIOS, MAX_NEW_TOKENS, NUM_LEVELS,
                    ABLATION_FUSION_MODES, resolve_image_path)
from utils.vlm_inference import resize_image, infer_single, is_correct

with open(os.path.join(DATA_DIR, "textvqa_val.json")) as f:
    val_samples = json.load(f)

def quick_eval_qvfp(model, samples):
    device = next(model.parameters()).device
    preds, gts, levels = [], [], []
    for s in tqdm(samples, desc="eval", leave=False):
        try:
            qe, ve = _get_eval_emb("textvqa_val", s, device)
            lv = model.predict(qe, ve).item()
        except Exception:
            lv = NUM_LEVELS - 1
        try:
            img  = Image.open(resolve_image_path(s["image_path"])).convert("RGB")
            pred = infer_single(vlm, processor,
                                resize_image(img, RESOLUTION_RATIOS[lv]),
                                s["question"], MAX_NEW_TOKENS)
        except Exception:
            pred = ""
        preds.append(pred); gts.append(s["answers"]); levels.append(lv)
    acc = round(sum(is_correct(p,g) for p,g in zip(preds,gts))/len(preds)*100, 2)
    return summarise("", acc, levels)

print("\n===== ABLATION 1: Fusion Mode =====")
fusion_results = []
for mode in ABLATION_FUSION_MODES:
    ckpt = os.path.join(CKPT_DIR, f"qvfp_sl_{mode}_3lvl_best.pt")
    if not os.path.exists(ckpt):
        print(f"  Training {mode}...")
        train_qvfp_sl(mode, 3)
    m   = load_qvfp_ckpt(ckpt)
    res = quick_eval_qvfp(m, val_samples)
    res["method"] = f"fusion_{mode}"
    fusion_results.append(res)
    print(f"  {mode:<14} | acc={res['vqa_accuracy']:.1f}% | "
          f"tokens={res['avg_tokens']:.0f} | red={res['token_reduction%']:.1f}%")

with open(os.path.join(RESULT_DIR, "ablation_fusion.json"), "w") as f:
    json.dump(fusion_results, f, indent=2)
print("Saved ablation_fusion.json")


In [ ]:
# ── Cell 6b: Ablation 2 — Resolution levels ────────────────────────────────
LEVEL_CONFIGS = {2: [0.25,1.0], 3: [0.25,0.5,1.0], 4: [0.25,0.33,0.67,1.0]}
from config import ABLATION_LEVEL_COUNTS, resolve_image_path

print("\n===== ABLATION 2: Resolution Levels =====")
levels_results = []
for n in ABLATION_LEVEL_COUNTS:
    ckpt = os.path.join(CKPT_DIR, f"qvfp_sl_concat_{n}lvl_best.pt")
    if not os.path.exists(ckpt):
        print(f"  Training {n}-level model...")
        train_qvfp_sl("concat", n)
    m      = load_qvfp_ckpt(ckpt)
    ratios = LEVEL_CONFIGS[n]
    device = next(m.parameters()).device
    preds, gts, levels_used = [], [], []
    for s in tqdm(val_samples, desc=f"{n}-level", leave=False):
        try:
            qe, ve = _get_eval_emb("textvqa_val", s, device)
            lv = min(m.predict(qe, ve).item(), n-1)
        except Exception:
            lv = n - 1
        try:
            img  = Image.open(resolve_image_path(s["image_path"])).convert("RGB")
            pred = infer_single(vlm, processor, resize_image(img, ratios[lv]),
                                s["question"], MAX_NEW_TOKENS)
        except Exception:
            pred = ""
        preds.append(pred); gts.append(s["answers"]); levels_used.append(lv)
    acc     = round(sum(is_correct(p,g) for p,g in zip(preds,gts))/len(preds)*100, 2)
    tok_map = {i: int(ratios[i]**2*1024) for i in range(n)}
    avg_t   = round(sum(tok_map[l] for l in levels_used)/len(levels_used), 1)
    red     = round((1-avg_t/1024)*100, 1)
    res     = {"method": f"{n}_levels", "vqa_accuracy": acc,
               "avg_tokens": avg_t, "token_reduction%": red}
    levels_results.append(res)
    print(f"  {n}-level | acc={acc:.1f}% | tokens={avg_t:.0f} | reduction={red:.1f}%")

with open(os.path.join(RESULT_DIR, "ablation_levels.json"), "w") as f:
    json.dump(levels_results, f, indent=2)
print("Saved ablation_levels.json")


In [ ]:
# ── Cell 6c: Ablation 3 — Query keyword analysis ───────────────────────────
KEYWORD_GROUPS = {
    "global_questions": ["is there","are there","how many","what color","what is","where is","who is"],
    "finegrained_text": ["read","what does it say","what text","what word","what number","what letters","what sign"],
    "counting":         ["how many","count","number of"],
    "attributes":       ["what color","what shape","what size","what type"],
}

print("\n===== ABLATION 3: Query Keyword Analysis =====")
sl_model = load_qvfp_ckpt(os.path.join(CKPT_DIR,"qvfp_sl_concat_3lvl_best.pt"))
device   = next(sl_model.parameters()).device
group_levels = {g: [] for g in KEYWORD_GROUPS}

for s in tqdm(val_samples, desc="keyword analysis", leave=False):
    try:
        qe, ve = _get_eval_emb("textvqa_val", s, device)
        lv = sl_model.predict(qe, ve).item()
    except Exception:
        continue
    q_lower = s["question"].lower()
    for group, kws in KEYWORD_GROUPS.items():
        if any(kw in q_lower for kw in kws):
            group_levels[group].append(lv)

kw_results = {}
for group, lvs in group_levels.items():
    if not lvs: continue
    avg = sum(lvs)/len(lvs)
    kw_results[group] = {
        "avg_level": round(avg,2),
        "low_pct":   round(lvs.count(0)/len(lvs)*100,1),
        "mid_pct":   round(lvs.count(1)/len(lvs)*100,1),
        "full_pct":  round(lvs.count(2)/len(lvs)*100,1),
        "n":         len(lvs)
    }
    print(f"  {group:<25} | avg_level={avg:.2f} | "
          f"low={kw_results[group]['low_pct']}% mid={kw_results[group]['mid_pct']}% "
          f"full={kw_results[group]['full_pct']}% (n={len(lvs)})")

with open(os.path.join(RESULT_DIR,"ablation_keywords.json"),"w") as f:
    json.dump(kw_results, f, indent=2)
print("Saved ablation_keywords.json")


## Section 7 — Generate all paper figures

Produces 6 PDF figures saved to `RESULT_DIR/figures/`.


In [ ]:
# ── Cell 7: Generate all figures ──────────────────────────────────────────
import json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from config import RESULT_DIR

FIG_DIR = os.path.join(RESULT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

STYLE = {
    "full_res":       {"c":"#888780","m":"s","l":"Full-res"},
    "fixed_025":      {"c":"#F09595","m":"^","l":"Fixed 1/4"},
    "fixed_050":      {"c":"#EF9F27","m":"^","l":"Fixed 1/2"},
    "oracle":         {"c":"#1D9E75","m":"D","l":"Oracle"},
    "qadc_sl":        {"c":"#534AB7","m":"o","l":"QADC-SL (ours)"},
    "qadc_rl_lam0.1": {"c":"#378ADD","m":"o","l":"QADC-RL λ=0.1"},
    "qadc_rl_lam0.3": {"c":"#185FA5","m":"o","l":"QADC-RL λ=0.3"},
    "qadc_rl_lam0.5": {"c":"#042C53","m":"o","l":"QADC-RL λ=0.5"},
}

def save(name):
    p = os.path.join(FIG_DIR, name)
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  Saved: {p}")

# Fig 1: Accuracy vs Token Reduction
p = os.path.join(RESULT_DIR,"main_results.json")
if os.path.exists(p):
    with open(p) as f: all_res = json.load(f)
    for sname, rows in all_res.items():
        fig, ax = plt.subplots(figsize=(7,5))
        for row in rows:
            st = STYLE.get(row["method"], {"c":"#aaa","m":"x","l":row["method"]})
            sz = 120 if "qadc" in row["method"] else 70
            ax.scatter(row["token_reduction%"], row["vqa_accuracy"],
                       s=sz, color=st["c"], marker=st["m"], zorder=3, edgecolors="white", linewidths=0.8)
            ax.annotate(st["l"], (row["token_reduction%"], row["vqa_accuracy"]),
                        textcoords="offset points", xytext=(6,3), fontsize=8, color=st["c"])
        ax.set_xlabel("Token reduction (%)"); ax.set_ylabel("VQA accuracy (%)")
        ax.set_title(f"Accuracy vs Efficiency — {sname}")
        ax.grid(True, alpha=0.3, linewidth=0.5); ax.spines[["top","right"]].set_visible(False)
        plt.tight_layout(); save(f"fig1_acc_vs_tokens_{sname}.pdf")

# Fig 2: Training curves
p = os.path.join(RESULT_DIR,"sl_curve_concat_3lvl.json")
if os.path.exists(p):
    with open(p) as f: data = json.load(f)
    hist = data["history"]
    fig, (ax1,ax2) = plt.subplots(1,2,figsize=(10,4))
    ax1.plot([h["epoch"] for h in hist],[h["train_loss"] for h in hist],color="#534AB7",linewidth=1.8)
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Training loss"); ax1.set_title("QVFP-SL training loss")
    ax1.grid(True,alpha=0.3,linewidth=0.5); ax1.spines[["top","right"]].set_visible(False)
    ax2.plot([h["epoch"] for h in hist],[h["val_acc"] for h in hist],color="#1D9E75",linewidth=1.8)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val accuracy (%)"); ax2.set_title("QVFP-SL validation accuracy")
    ax2.grid(True,alpha=0.3,linewidth=0.5); ax2.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); save("fig2_training_curves.pdf")

# Fig 3: Fusion ablation
p = os.path.join(RESULT_DIR,"ablation_fusion.json")
if os.path.exists(p):
    with open(p) as f: rows = json.load(f)
    labels=[r["method"].replace("fusion_","") for r in rows]
    accs=[r["vqa_accuracy"] for r in rows]; toks=[r["token_reduction%"] for r in rows]
    x=np.arange(len(labels)); w=0.35
    fig,ax=plt.subplots(figsize=(7,4))
    b1=ax.bar(x-w/2,accs,w,label="VQA accuracy (%)",color="#534AB7",alpha=0.85)
    b2=ax.bar(x+w/2,toks,w,label="Token reduction (%)",color="#1D9E75",alpha=0.85)
    ax.bar_label(b1,fmt="%.1f",fontsize=8,padding=2); ax.bar_label(b2,fmt="%.1f",fontsize=8,padding=2)
    ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=10); ax.set_ylabel("Percentage (%)")
    ax.legend(); ax.set_title("Ablation: Fusion Mode"); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); save("fig3_ablation_fusion.pdf")

# Fig 4: Levels ablation
p = os.path.join(RESULT_DIR,"ablation_levels.json")
if os.path.exists(p):
    with open(p) as f: rows = json.load(f)
    labels=[r["method"] for r in rows]
    accs=[r["vqa_accuracy"] for r in rows]; toks=[r["token_reduction%"] for r in rows]
    x=np.arange(len(labels)); w=0.35
    fig,ax=plt.subplots(figsize=(6,4))
    b1=ax.bar(x-w/2,accs,w,label="VQA accuracy (%)",color="#534AB7",alpha=0.85)
    b2=ax.bar(x+w/2,toks,w,label="Token reduction (%)",color="#EF9F27",alpha=0.85)
    ax.bar_label(b1,fmt="%.1f",fontsize=9,padding=2); ax.bar_label(b2,fmt="%.1f",fontsize=9,padding=2)
    ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=10); ax.set_ylabel("Percentage (%)")
    ax.legend(); ax.set_title("Ablation: Resolution Levels"); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); save("fig4_ablation_levels.pdf")

# Fig 5: Keyword analysis
p = os.path.join(RESULT_DIR,"ablation_keywords.json")
if os.path.exists(p):
    with open(p) as f: data = json.load(f)
    groups=list(data.keys()); low=[data[g]["low_pct"] for g in groups]
    mid=[data[g]["mid_pct"] for g in groups]; full=[data[g]["full_pct"] for g in groups]
    x=np.arange(len(groups))
    fig,ax=plt.subplots(figsize=(8,4))
    ax.bar(x,low,label="Low (1/4×)",color="#1D9E75",alpha=0.85)
    ax.bar(x,mid,bottom=low,label="Mid (1/2×)",color="#EF9F27",alpha=0.85)
    ax.bar(x,full,bottom=[l+m for l,m in zip(low,mid)],label="Full (1×)",color="#534AB7",alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels([g.replace("_","\n") for g in groups],fontsize=9)
    ax.set_ylabel("Percentage (%)"); ax.legend(loc="upper right")
    ax.set_title("Level Distribution by Query Type"); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); save("fig5_keyword_analysis.pdf")

# Fig 6: SL vs RL
p = os.path.join(RESULT_DIR,"main_results.json")
if os.path.exists(p):
    with open(p) as f: all_res = json.load(f)
    rows = all_res.get("textvqa_val", list(all_res.values())[0])
    rm   = {r["method"]:r for r in rows}
    mnames=["qadc_sl"]+[f"qadc_rl_lam{l}" for l in [0.1,0.3,0.5]]
    labels=["SL","RL λ=0.1","RL λ=0.3","RL λ=0.5"]
    accs=[rm.get(m,{}).get("vqa_accuracy",0) for m in mnames]
    toks=[rm.get(m,{}).get("token_reduction%",0) for m in mnames]
    x=np.arange(len(labels)); w=0.35
    fig,ax=plt.subplots(figsize=(7,4))
    b1=ax.bar(x-w/2,accs,w,label="VQA accuracy (%)",color="#534AB7",alpha=0.85)
    b2=ax.bar(x+w/2,toks,w,label="Token reduction (%)",color="#D85A30",alpha=0.85)
    ax.bar_label(b1,fmt="%.1f",fontsize=9,padding=2); ax.bar_label(b2,fmt="%.1f",fontsize=9,padding=2)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Percentage (%)")
    ax.legend(); ax.set_title("SL vs RL Training"); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); save("fig6_sl_vs_rl.pdf")

print(f"\nAll figures → {FIG_DIR}")


## Section 8 — Single-sample sanity check

Quick interactive test to verify the full pipeline works end-to-end.


In [ ]:
# ── Cell 8: Single sample demo ─────────────────────────────────────────────
import torch, os, json
from PIL import Image
from config import CKPT_DIR, LABEL_DIR, DATA_DIR, RESOLUTION_RATIOS, LEVEL_NAMES, resolve_image_path
from utils.vlm_inference import resize_image, infer_single, extract_embeddings, is_correct

ckpt       = torch.load(os.path.join(CKPT_DIR, "qvfp_sl_concat_3lvl_best.pt"),
                        weights_only=False)
demo_model = QVFP(ckpt.get("fusion_mode","concat"), ckpt.get("num_levels",3))
demo_model.load_state_dict(ckpt["model_state"]); demo_model.eval()
device     = "cuda" if torch.cuda.is_available() else "cpu"
demo_model = demo_model.to(device)

with open(os.path.join(DATA_DIR, "textvqa_val.json")) as f:
    val_s = json.load(f)

for sample in val_s[:10]:
    try:
        image_path = resolve_image_path(sample["image_path"])
        image      = Image.open(image_path).convert("RGB")
        query      = sample["question"]
        gts        = sample["answers"]
        q_emb, v_emb = extract_embeddings(vlm, processor, image, query)
        level  = demo_model.predict(q_emb.unsqueeze(0).to(device),
                                    v_emb.unsqueeze(0).to(device)).item()
        ratio  = RESOLUTION_RATIOS[level]
        answer = infer_single(vlm, processor, resize_image(image, ratio), query)
        correct = is_correct(answer, gts)
        print(f"Query   : {query}")
        print(f"GT      : {gts[:3]}")
        print(f"Level   : {LEVEL_NAMES[level]} (ratio={ratio}, ~{[64,256,1024][level]} tokens)")
        print(f"Answer  : {answer}")
        print(f"Correct : {'Yes' if correct else 'No'}")
        break
    except Exception as e:
        print(f"  Skipping: {e}")
